# NB-02-1 $SU(2)$ の指数関数：$e^{-i\theta\,\hat n\cdot\vec\sigma/2}$ を実装する

[パウリ行列のノート](../../研究ノート/04_群論・代数/pauli_matrices_derivation_and_group.md#p7)の Part VII §2 で導いた閉じた形

$$
e^{-i\frac\theta2\,\hat n\cdot\vec\sigma}=\cos\frac\theta2\,I-i\sin\frac\theta2\,\hat n\cdot\vec\sigma
$$

を実装し、行列の指数関数の定義（級数）と比べます。さらに、$SU(2)$ の元から回転軸と角度を取り出す計算、Qiskit の回転ゲート（`RXGate` など）との関係、二重被覆（$\theta=2\pi$ で $-I$）を確かめます。$R_z,R_y$ の具体形は、[ユニタリ行列のノート](../../研究ノート/04_群論・代数/unitary_matrix_full_decomposition.md)の Part VIII でも計算しています。

**使い方**：上から順に実行します。各セルの `assert` が、ノートの式と計算結果の一致を確かめます。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from numpy.typing import NDArray
from scipy.linalg import expm

np.set_printoptions(precision=4, suppress=True)

# 型の別名（型ヒントを読みやすくするため。実行時には何も確かめない）
Matrix = NDArray[np.complex128]   # 2×2 の複素行列
Ket = NDArray[np.complex128]      # 長さ 2 の複素ベクトル（状態ベクトル）
Vector = NDArray[np.float64]      # 3次元の実ベクトル
Rot3 = NDArray[np.float64]        # 3×3 の実行列（3次元の回転）

sx: Matrix = np.array([[0, 1], [1, 0]], dtype=complex)
sy: Matrix = np.array([[0, -1j], [1j, 0]])
sz: Matrix = np.array([[1, 0], [0, -1]], dtype=complex)
S: tuple[Matrix, Matrix, Matrix] = (sx, sy, sz)
I2: Matrix = np.eye(2, dtype=complex)


def dot_sigma(v: Vector) -> Matrix:
    return v[0] * sx + v[1] * sy + v[2] * sz


def unit(theta: float, phi: float) -> Vector:
    """球座標の単位ベクトル（θ：北極からの角度、φ：経度。ラジアン）"""
    return np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])


def U_rot(n: Vector, theta: float) -> Matrix:
    """ノートの Part VII §2 の閉じた形 exp(−iθ n·σ/2) = cos(θ/2) I − i sin(θ/2) n·σ（n は単位ベクトル）"""
    return np.cos(theta / 2) * I2 - 1j * np.sin(theta / 2) * dot_sigma(n)


def Rz(a: float) -> Matrix:
    return U_rot(np.array([0.0, 0.0, 1.0]), a)


def Ry(a: float) -> Matrix:
    return U_rot(np.array([0.0, 1.0, 0.0]), a)


def Rx(a: float) -> Matrix:
    return U_rot(np.array([1.0, 0.0, 0.0]), a)

## 1. 閉じた形と、級数の定義を比べる

行列の指数関数の定義は級数 $e^X=\sum_{m=0}^\infty\frac{X^m}{m!}$ です。$X=-i\frac\theta2\hat n\cdot\vec\sigma$ について、$N$ 項までの部分和と閉じた形の差（行列のノルム）を描きます。$(\hat n\cdot\vec\sigma)^2=I$ なので、偶数次と奇数次がそれぞれ $\cos\frac\theta2$、$\sin\frac\theta2$ のテイラー展開になり、$N$ を増やすと差は $0$ に近づきます。scipy の `expm` とも比べます。

In [ ]:
def partial_sum(X: Matrix, N: int) -> Matrix:
    """級数 Σ_{m=0}^{N} X^m / m! の部分和"""
    total, term = np.zeros_like(X), np.eye(2, dtype=complex)
    for m in range(N + 1):
        total = total + term
        term = term @ X / (m + 1)
    return total


rng = np.random.default_rng(0)
for _ in range(50):
    n = rng.normal(size=3)
    n /= np.linalg.norm(n)
    th = rng.uniform(-10, 10)
    X = -1j * th / 2 * dot_sigma(n)
    assert np.allclose(U_rot(n, th), expm(X))              # 閉じた形 = scipy の expm
    assert np.allclose(U_rot(n, th), partial_sum(X, 60))   # 閉じた形 = 級数（60 項）

n0 = unit(1.0, 0.4)
fig = go.Figure()
for th in (1.0, 3.0, 6.0):
    errs = [np.linalg.norm(partial_sum(-1j * th / 2 * dot_sigma(n0), N) - U_rot(n0, th)) for N in range(0, 26)]
    fig.add_trace(go.Scatter(x=list(range(0, 26)), y=errs, mode="lines+markers", name=f"θ = {th}"))
fig.update_layout(height=380, width=700, yaxis_type="log", xaxis_title="部分和の項数 N", yaxis_title="閉じた形との差（ノルム）",
                  margin=dict(t=20))
fig

## 2. $SU(2)$ の元から、回転軸と角度を取り出す

閉じた形 $U=\cos\frac\theta2I-i\sin\frac\theta2\,\hat n\cdot\vec\sigma$ に、パウリ行列の直交性（ノートの Part II §4）を使うと、

$$
\cos\frac\theta2=\frac12\operatorname{tr}U,\qquad \sin\frac\theta2\;n_k=\frac i2\operatorname{tr}(\sigma_kU)
$$

で、$U$ から $\theta\in[0,2\pi]$ と $\hat n$ を取り出せます（ノートの Part I §4「どの元も $e^{iA}$ と書ける」の具体的な計算です）。ランダムな $SU(2)$ の元で、取り出した $(\theta,\hat n)$ から $U$ が元に戻ることを確かめます。

In [ ]:
from qiskit.quantum_info import random_unitary


def to_su2(V: Matrix) -> Matrix:
    """U(2) の元を、大域位相で割って SU(2) の元にする（det = 1）"""
    return V / np.sqrt(np.linalg.det(V))


def axis_angle(U: Matrix) -> tuple[Vector, float]:
    """SU(2) の元 U から、回転軸 n と角度 θ ∈ [0, 2π] を取り出す"""
    c = np.clip(np.trace(U).real / 2, -1, 1)
    theta = 2 * np.arccos(c)
    if np.isclose(np.sin(theta / 2), 0):
        return np.array([0.0, 0.0, 1.0]), theta          # U = ±I：軸は決まらない（どの軸でもよい）
    n = np.array([(0.5j * np.trace(Sk @ U)).real for Sk in S]) / np.sin(theta / 2)
    return n, theta


for seed in range(100):
    U = to_su2(random_unitary(2, seed=seed).data)
    assert np.allclose(U.conj().T @ U, I2) and np.isclose(np.linalg.det(U), 1)
    assert np.isclose(U[1, 1], np.conj(U[0, 0])) and np.isclose(U[0, 1], -np.conj(U[1, 0]))   # [[a, −b*], [b, a*]]
    n, th = axis_angle(U)
    assert np.isclose(np.linalg.norm(n), 1) and np.allclose(U_rot(n, th), U)

print("例：U =\n", U)
print(f"回転軸 n = {n}、角度 θ = {np.degrees(th):.2f}°")

## 3. Qiskit の回転ゲートとの関係

Qiskit の `RXGate(θ)`、`RYGate(θ)`、`RZGate(θ)` は、そのまま $e^{-i\theta\sigma/2}$ です。一方、よく使う `PhaseGate(λ)`（$P(\lambda)=\operatorname{diag}(1,e^{i\lambda})$）と `UGate(θ, φ, λ)` は、$R_z,R_y$ と**大域位相だけ**違います：

$$
P(\lambda)=e^{i\lambda/2}R_z(\lambda),\qquad U(\theta,\phi,\lambda)=e^{i(\phi+\lambda)/2}\,R_z(\phi)\,R_y(\theta)\,R_z(\lambda)
$$

大域位相はブロッホ球の上では見えないので、$P(\lambda)$ と $R_z(\lambda)$ は同じ回転を表します（$\det P(\lambda)=e^{i\lambda}$ なので、$P(\lambda)$ は $SU(2)$ の元ではありません）。

In [ ]:
from qiskit.circuit.library import PhaseGate, RXGate, RYGate, RZGate, UGate

for th in np.linspace(-7, 7, 15):
    assert np.allclose(RXGate(th).to_matrix(), Rx(th))
    assert np.allclose(RYGate(th).to_matrix(), Ry(th))
    assert np.allclose(RZGate(th).to_matrix(), Rz(th))
    assert np.allclose(PhaseGate(th).to_matrix(), np.exp(1j * th / 2) * Rz(th))

for t, p, l in rng.uniform(-np.pi, np.pi, size=(30, 3)):
    assert np.allclose(UGate(t, p, l).to_matrix(), np.exp(1j * (p + l) / 2) * Rz(p) @ Ry(t) @ Rz(l))

lam = 0.9
print("P(λ) =\n", PhaseGate(lam).to_matrix(), "\ndet P(λ) =", np.round(np.linalg.det(PhaseGate(lam).to_matrix()), 4),
      " ← e^{iλ}。大域位相 e^{iλ/2} を除くと R_z(λ)")

## 4. 二重被覆：$\theta=2\pi$ で $-I$

閉じた形で $\theta=2\pi$ とすると $U=\cos\pi\,I-i\sin\pi\,\hat n\cdot\vec\sigma=-I$ で、$I$ に戻るのは $\theta=4\pi$ です。どの軸でも同じであることを確かめ、$\frac12\operatorname{tr}U(\theta)=\cos\frac\theta2$（周期 $4\pi$）を描きます。ブロッホ球の上の回転（次のノートブック）は、周期 $2\pi$ です。

In [ ]:
for seed in range(20):
    n = rng.normal(size=3)
    n /= np.linalg.norm(n)
    assert np.allclose(U_rot(n, 2 * np.pi), -I2) and np.allclose(U_rot(n, 4 * np.pi), I2)
    assert np.allclose(RXGate(2 * np.pi).to_matrix(), -I2)

thetas = np.linspace(0, 4 * np.pi, 400)
fig2 = go.Figure([go.Scatter(x=thetas / np.pi, y=[np.trace(U_rot(n0, t)).real / 2 for t in thetas], mode="lines",
                             name="½ tr U(θ) = cos(θ/2)")])
fig2.add_vline(x=2, line_dash="dot", annotation_text="θ = 2π：U = −I")
fig2.add_vline(x=4, line_dash="dot", annotation_text="θ = 4π：U = I")
fig2.update_layout(height=360, width=700, xaxis_title="θ / π", yaxis_title="½ tr U", margin=dict(t=30))
fig2

## 観察のポイント

- 級数の部分和は、$\theta$ が大きいほど収束に多くの項が必要ですが、閉じた形は $\theta$ によらず2項（$I$ と $\hat n\cdot\vec\sigma$）で済みます。$(\hat n\cdot\vec\sigma)^2=I$ のおかげです。
- $SU(2)$ のどの元も、軸 $\hat n$ と角度 $\theta$ の組で書けます（$\theta$ を $4\pi-\theta$、$\hat n$ を $-\hat n$ に変えても同じ元）。
- Qiskit の `PhaseGate` と `RZGate` は大域位相だけ違い、同じ回転を表します。回路を組むとき（Phase 2）に、どちらを使っても測定結果は変わりません。

**次のノートブック**：[02_bloch_rotation.ipynb](02_bloch_rotation.ipynb) で、$U$ がブロッホ球をどう回すかを、スライダーで見ます。